In [ ]:
# Glacier Segmentation Training - SMART FILE MATCHING VERSION

# 1. Install dependencies
print("Installing dependencies...")
!pip install -q segmentation-models-pytorch albumentations optuna rasterio opencv-python-headless plotly

# 2. Clone your GitHub repository
print("Cloning repository...")
import os
import sys

if not os.path.exists('/kaggle/working/Glacier-Hack-Own'):
    !git clone -q https://github.com/satya-18-w/Glacier-Hack-Own.git

# 3. Set up paths
os.chdir('/kaggle/working/Glacier-Hack-Own')
sys.path.insert(0, '/kaggle/working/Glacier-Hack-Own')

# 4. Import required libraries
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import importlib

# 5. First, let's understand the file structure better
print("=== ANALYZING FILE STRUCTURE ===")
dataset_path = '/kaggle/input/glacer/Train'

def analyze_file_patterns(dataset_path, bands, label_dir):
    """Analyze file naming patterns to understand how to match files"""
    print("File pattern analysis:")
    
    for band in bands + [label_dir]:
        band_path = os.path.join(dataset_path, band)
        if os.path.exists(band_path):
            files = os.listdir(band_path)
            if files:
                print(f"\n{band} files (first 3):")
                for f in files[:3]:
                    print(f"  {f}")
                
                # Analyze naming pattern
                sample_file = files[0]
                parts = sample_file.split('_')
                print(f"  Pattern: {parts}")

analyze_file_patterns(dataset_path, ['Band1', 'Band2', 'Band3', 'Band4', 'Band5'], 'label')

# 6. Import the fixed dataset module
print("\n=== LOADING DATASET ===")
from data.dataset import GlacierDataset
from data.transforms import get_train_transforms, get_val_transforms
from models.model_factory import create_model
from models.losses import get_loss
from training.trainer import GlacierTrainer
from training.optimizer import get_optimizer

# 7. Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 8. Create dataset with smart file matching
bands = ['Band1', 'Band2', 'Band3', 'Band4', 'Band5']
label_dir = 'label'

print("Creating dataset with smart file matching...")
dataset = GlacierDataset(
    root_dir=dataset_path,
    bands=bands,
    label_dir=label_dir,
    transform=get_train_transforms((256, 256)),
    image_size=(256, 256)
)

print(f"Dataset created with {len(dataset)} samples")

if len(dataset) > 0:
    # Test one sample
    sample_img, sample_mask = dataset[0]
    print(f"Sample image shape: {sample_img.shape}")
    print(f"Sample mask shape: {sample_mask.shape}")
    print(f"Image range: [{sample_img.min():.3f}, {sample_img.max():.3f}]")
    print(f"Mask unique values: {torch.unique(sample_mask)}")

# 9. Split dataset
train_idx, val_idx = train_test_split(
    list(range(len(dataset))), 
    train_size=0.8, 
    random_state=42,
    shuffle=True
)

train_dataset = Subset(dataset, train_idx)
val_dataset = Subset(dataset, val_idx)

print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")

# 10. Create data loaders
batch_size = 4
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

# 11. Test data loader
print("Testing data loader...")
try:
    for i, (images, masks) in enumerate(train_loader):
        print(f"Batch {i}: images shape: {images.shape}, masks shape: {masks.shape}")
        if i >= 1:  # Test a couple of batches
            break
    print("✅ Data loader works!")
except Exception as e:
    print(f"❌ Error in data loader: {e}")

# 12. Create model
print("Creating model...")
model = create_model('unet', {
    'encoder_name': 'resnet34',
    'encoder_weights': None,
    'in_channels': len(bands),
    'classes': 1
})
model = model.to(device)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# 13. Setup training
optimizer = get_optimizer('adam', model.parameters(), lr=0.001)
loss_fn = get_loss('bce_dice')

trainer = GlacierTrainer(model, optimizer, loss_fn, device)

# 14. Training loop
print("Starting training...")
epochs = 10
best_mcc = 0.0

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    
    # Training
    train_loss = trainer.train_epoch(train_loader)
    
    # Validation
    val_loss, val_mcc = trainer.validate_epoch(val_loader)
    
    print(f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}, Val MCC: {val_mcc:.4f}")
    
    # Save best model
    if val_mcc > best_mcc:
        best_mcc = val_mcc
        torch.save(model.state_dict(), f'/kaggle/working/best_model_epoch_{epoch+1}_mcc_{val_mcc:.4f}.pth')
        print(f"✅ New best model saved with MCC: {val_mcc:.4f}")

print(f"🎉 Training completed! Best MCC: {best_mcc:.4f}")

# 15. Save final model
torch.save(model.state_dict(), '/kaggle/working/final_model.pth')
print("✅ Final model saved")

print("📊 Script execution completed!")

Installing dependencies...
